In [ ]:
"""
colab_train_neurosense.py
=========================
Paste this entire file into ONE Google Colab cell and run.

This script expects you already ran local preprocessing and uploaded:
  /content/drive/MyDrive/neurosense_preprocessed/neurosense_features_256hz.npz

What it does:
1. Loads precomputed NeuroSense features.
2. Trains and evaluates a strong stacked ensemble.
3. Reports:
   - Subject-dependent (session-safe) metrics
   - Subject-independent (participant-safe) metrics
4. Simulates personalization with 10 calibration sessions per user.
5. Saves deployable model artifacts.
"""

# =============================================================================
# INSTALL
# =============================================================================
import subprocess
import sys

for pkg in [
    "numpy",
    "pandas",
    "scikit-learn",
    "imbalanced-learn",
    "xgboost",
    "optuna",
    "seaborn",
    "matplotlib",
    "joblib",
]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=True)


# =============================================================================
# IMPORTS
# =============================================================================
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from sklearn.isotonic import IsotonicRegression

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

from google.colab import drive
drive.mount("/content/drive")


# =============================================================================
# CONFIG
# =============================================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NPZ_PATH = Path("/content/drive/MyDrive/neurosense_preprocessed/neurosense_features_256hz.npz")
SAVE_DIR = Path("/content/drive/MyDrive/neurosense_trained")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_TRIALS = 25
CAL_SESSIONS_PER_USER = 10


# =============================================================================
# LOAD FEATURES
# =============================================================================
print("=" * 70)
print("STEP 1  Loading preprocessed NPZ")
print("=" * 70)

if not NPZ_PATH.exists():
    raise FileNotFoundError(f"NPZ not found: {NPZ_PATH}")

data = np.load(NPZ_PATH, allow_pickle=True)

if "features" in data:
    F = data["features"].astype(np.float32)
elif "F" in data:
    F = data["F"].astype(np.float32)
else:
    raise KeyError("Expected key 'features' (or 'F') in NPZ")

if "labels" in data:
    y = data["labels"].astype(np.int8)
elif "y" in data:
    y = data["y"].astype(np.int8)
else:
    raise KeyError("Expected key 'labels' (or 'y') in NPZ")

if "subject_id" in data:
    subjects = data["subject_id"].astype(str)
elif "participant_id" in data:
    subjects = data["participant_id"].astype(str)
else:
    raise KeyError("Expected key 'subject_id' (or 'participant_id') in NPZ")

if "session_id" in data:
    sessions = data["session_id"].astype(str)
else:
    raise KeyError("Expected key 'session_id' in NPZ")

F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)

bc = np.bincount(y)
print(f"Features shape      : {F.shape}")
print(f"Participants        : {len(np.unique(subjects))}")
print(f"Sessions            : {len(np.unique(sessions))}")
print(f"Positive (0)        : {bc[0]:,} ({bc[0]/len(y)*100:.1f}%)")
print(f"Negative (1)        : {bc[1]:,} ({bc[1]/len(y)*100:.1f}%)")


# =============================================================================
# SPLITS
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2  Building splits")
print("=" * 70)

# Subject-dependent but session-safe (no same session leakage)
gss_session = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_dep, idx_test_dep = next(gss_session.split(F, y, groups=sessions))

gss_session_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_dep_rel, idx_val_dep_rel = next(
    gss_session_val.split(F[idx_train_dep], y[idx_train_dep], groups=sessions[idx_train_dep])
)
idx_train_dep_final = idx_train_dep[idx_train_dep_rel]
idx_val_dep_final = idx_train_dep[idx_val_dep_rel]

# Subject-independent (no participant leakage)
gss_sub = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_ind, idx_test_ind = next(gss_sub.split(F, y, groups=subjects))

gss_sub_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_ind_rel, idx_val_ind_rel = next(
    gss_sub_val.split(F[idx_train_ind], y[idx_train_ind], groups=subjects[idx_train_ind])
)
idx_train_ind_final = idx_train_ind[idx_train_ind_rel]
idx_val_ind_final = idx_train_ind[idx_val_ind_rel]

print("Subject-dependent (session-safe):")
print(f"  train={len(idx_train_dep_final):,}, val={len(idx_val_dep_final):,}, test={len(idx_test_dep):,}")
print("Subject-independent (participant-safe):")
print(f"  train={len(idx_train_ind_final):,}, val={len(idx_val_ind_final):,}, test={len(idx_test_ind):,}")
print(f"  Train participants={len(np.unique(subjects[idx_train_ind_final]))}, "
      f"Test participants={len(np.unique(subjects[idx_test_ind]))}")


# =============================================================================
# MODEL HELPERS
# =============================================================================
def tune_xgb_with_optuna(X_train, y_train, n_trials=25):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "eval_metric": "auc",
            "random_state": RANDOM_SEED,
            "n_jobs": -1,
            "verbosity": 0,
        }
        clf = xgb.XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
        score = cross_val_score(clf, X_train, y_train, cv=cv,
                                scoring="balanced_accuracy", n_jobs=-1)
        return score.mean()

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value


def build_stacked_model(best_xgb_params):
    xgb_clf = xgb.XGBClassifier(
        n_estimators=best_xgb_params["n_estimators"],
        max_depth=best_xgb_params["max_depth"],
        min_child_weight=best_xgb_params["min_child_weight"],
        learning_rate=best_xgb_params["learning_rate"],
        subsample=best_xgb_params["subsample"],
        colsample_bytree=best_xgb_params["colsample_bytree"],
        reg_alpha=best_xgb_params["reg_alpha"],
        reg_lambda=best_xgb_params["reg_lambda"],
        gamma=best_xgb_params["gamma"],
        eval_metric="auc",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0,
    )

    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

    lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")

    svm = CalibratedClassifierCV(
        LinearSVC(C=0.2, max_iter=5000, class_weight="balanced", random_state=RANDOM_SEED),
        cv=3,
    )

    meta = LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )

    model = StackingClassifier(
        estimators=[("xgb", xgb_clf), ("rf", rf), ("lda", lda), ("svm", svm)],
        final_estimator=meta,
        passthrough=True,
        cv=5,
        n_jobs=-1,
    )
    return model


def fit_eval_split(name, idx_tr, idx_va, idx_te):
    print("\n" + "-" * 70)
    print(f"{name}")
    print("-" * 70)

    X_tr, y_tr = F[idx_tr], y[idx_tr]
    X_va, y_va = F[idx_va], y[idx_va]
    X_te, y_te = F[idx_te], y[idx_te]

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_va_sc = scaler.transform(X_va)
    X_te_sc = scaler.transform(X_te)

    smt = SMOTETomek(smote=SMOTE(k_neighbors=3, random_state=RANDOM_SEED),
                     random_state=RANDOM_SEED)
    X_tr_res, y_tr_res = smt.fit_resample(X_tr_sc, y_tr)

    print(f"train class before: {np.bincount(y_tr)}")
    print(f"train class after : {np.bincount(y_tr_res)}")

    print("Tuning XGBoost with Optuna...")
    best_params, best_cv = tune_xgb_with_optuna(X_tr_res, y_tr_res, n_trials=OPTUNA_TRIALS)
    print(f"best CV BA: {best_cv*100:.2f}%")

    model = build_stacked_model(best_params)
    model.fit(X_tr_res, y_tr_res)

    y_va_prob = model.predict_proba(X_va_sc)[:, 1]
    y_va_pred = (y_va_prob >= 0.5).astype(np.int8)
    va_ba = balanced_accuracy_score(y_va, y_va_pred)
    va_auc = roc_auc_score(y_va, y_va_prob)

    y_te_prob = model.predict_proba(X_te_sc)[:, 1]
    y_te_pred = (y_te_prob >= 0.5).astype(np.int8)
    te_ba = balanced_accuracy_score(y_te, y_te_pred)
    te_auc = roc_auc_score(y_te, y_te_prob)
    te_acc = (y_te_pred == y_te).mean()

    print("Validation:")
    print(f"  BA   : {va_ba*100:.2f}%")
    print(f"  AUC  : {va_auc:.4f}")
    print("Test:")
    print(f"  Acc  : {te_acc*100:.2f}%")
    print(f"  BA   : {te_ba*100:.2f}%")
    print(f"  AUC  : {te_auc:.4f}")
    print(classification_report(y_te, y_te_pred, target_names=["Positive(0)", "Negative(1)"]))

    cm = confusion_matrix(y_te, y_te_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["Pos", "Neg"], yticklabels=["Pos", "Neg"],
                cbar=False, ax=ax)
    ax.set_title(f"{name} - Test Confusion")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    fig.savefig(SAVE_DIR / f"cm_{name.replace(' ', '_').lower()}.png", dpi=150)
    plt.close(fig)

    return {
        "name": name,
        "model": model,
        "scaler": scaler,
        "best_xgb": best_params,
        "val_ba": va_ba,
        "val_auc": va_auc,
        "test_ba": te_ba,
        "test_auc": te_auc,
        "test_acc": te_acc,
        "idx_test": idx_te,
    }


# =============================================================================
# TRAIN / TEST
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3  Train and evaluate")
print("=" * 70)

res_dep = fit_eval_split(
    "Subject-Dependent Session-Safe",
    idx_train_dep_final,
    idx_val_dep_final,
    idx_test_dep,
)

res_ind = fit_eval_split(
    "Subject-Independent Participant-Safe",
    idx_train_ind_final,
    idx_val_ind_final,
    idx_test_ind,
)


# =============================================================================
# PERSONALIZATION SIMULATION
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4  Personalization simulation")
print("=" * 70)

global_model = res_ind["model"]
global_scaler = res_ind["scaler"]

idx_te = res_ind["idx_test"]
X_te = F[idx_te]
y_te = y[idx_te]
sub_te = subjects[idx_te]
sess_te = sessions[idx_te]

X_te_sc = global_scaler.transform(X_te)
p_global = global_model.predict_proba(X_te_sc)[:, 1]

rows = []
for sub in np.unique(sub_te):
    m_sub = sub_te == sub
    if m_sub.sum() < 100:
        continue

    sess = sorted(np.unique(sess_te[m_sub]))
    if len(sess) <= CAL_SESSIONS_PER_USER:
        continue

    cal_sess = set(sess[:CAL_SESSIONS_PER_USER])
    eval_sess = set(sess[CAL_SESSIONS_PER_USER:])

    m_cal = m_sub & np.isin(sess_te, list(cal_sess))
    m_eval = m_sub & np.isin(sess_te, list(eval_sess))
    if m_cal.sum() < 30 or m_eval.sum() < 30:
        continue

    y_cal = y_te[m_cal]
    y_eval = y_te[m_eval]
    p_cal = p_global[m_cal]
    p_eval = p_global[m_eval]

    if len(np.unique(y_cal)) < 2 or len(np.unique(y_eval)) < 2:
        continue

    before = balanced_accuracy_score(y_eval, (p_eval >= 0.5).astype(np.int8))

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_cal, y_cal)
    after_iso = balanced_accuracy_score(y_eval, (iso.predict(p_eval) >= 0.5).astype(np.int8))

    X_cal_ad = np.column_stack([p_cal, p_cal ** 2])
    X_eval_ad = np.column_stack([p_eval, p_eval ** 2])
    lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
    lr.fit(X_cal_ad, y_cal)
    p_eval_lr = lr.predict_proba(X_eval_ad)[:, 1]
    after_lr = balanced_accuracy_score(y_eval, (p_eval_lr >= 0.5).astype(np.int8))

    rows.append({
        "subject": sub,
        "ba_before": float(before),
        "ba_after": float(max(after_iso, after_lr)),
        "best_adapter": "isotonic" if after_iso >= after_lr else "logistic",
        "n_cal_windows": int(m_cal.sum()),
        "n_eval_windows": int(m_eval.sum()),
    })

if rows:
    df = pd.DataFrame(rows).sort_values("ba_after", ascending=False)
    df.to_csv(SAVE_DIR / "personalization_per_user.csv", index=False)
    print(df.to_string(index=False))

    before_mean = df["ba_before"].mean() * 100
    after_mean = df["ba_after"].mean() * 100
    print(f"\nMean BA before: {before_mean:.2f}%")
    print(f"Mean BA after : {after_mean:.2f}%")
    print(f"Mean gain     : {after_mean - before_mean:+.2f}%")
else:
    before_mean = None
    after_mean = None
    print("Not enough eligible users for stable personalization simulation.")


# =============================================================================
# TRAIN FINAL MODEL ON ALL DATA + SAVE
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5  Train final global model and save")
print("=" * 70)

best_params_final = res_ind["best_xgb"]

scaler_final = StandardScaler()
F_sc_all = scaler_final.fit_transform(F)

smt = SMOTETomek(smote=SMOTE(k_neighbors=3, random_state=RANDOM_SEED), random_state=RANDOM_SEED)
F_res, y_res = smt.fit_resample(F_sc_all, y)

model_final = build_stacked_model(best_params_final)
model_final.fit(F_res, y_res)

joblib.dump(model_final, SAVE_DIR / "neurosense_global_model.pkl")
joblib.dump(scaler_final, SAVE_DIR / "neurosense_global_scaler.pkl")

meta = {
    "npz_path": str(NPZ_PATH),
    "n_windows": int(len(F)),
    "n_features": int(F.shape[1]),
    "n_subjects": int(len(np.unique(subjects))),
    "n_sessions": int(len(np.unique(sessions))),
    "subject_dependent_test_ba": float(res_dep["test_ba"]),
    "subject_independent_test_ba": float(res_ind["test_ba"]),
    "subject_independent_test_auc": float(res_ind["test_auc"]),
    "best_xgb_params": best_params_final,
}
if before_mean is not None:
    meta["personalization_before_mean_ba"] = float(before_mean / 100.0)
    meta["personalization_after_mean_ba"] = float(after_mean / 100.0)

with open(SAVE_DIR / "neurosense_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:")
for fp in [
    SAVE_DIR / "neurosense_global_model.pkl",
    SAVE_DIR / "neurosense_global_scaler.pkl",
    SAVE_DIR / "neurosense_model_metadata.json",
]:
    print(f"  {fp} ({fp.stat().st_size/1024:.1f} KB)")


# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print("Subject-dependent (session-safe):")
print(f"  Acc : {res_dep['test_acc']*100:.2f}%")
print(f"  BA  : {res_dep['test_ba']*100:.2f}%")
print(f"  AUC : {res_dep['test_auc']:.4f}")

print("\nSubject-independent (participant-safe, realistic):")
print(f"  Acc : {res_ind['test_acc']*100:.2f}%")
print(f"  BA  : {res_ind['test_ba']*100:.2f}%")
print(f"  AUC : {res_ind['test_auc']:.4f}")

if before_mean is not None:
    print("\nPersonalization simulation (10 sessions):")
    print(f"  Before mean BA : {before_mean:.2f}%")
    print(f"  After  mean BA : {after_mean:.2f}%")
    print(f"  Mean gain      : {after_mean - before_mean:+.2f}%")

print("\nDone.")


Mounted at /content/drive
STEP 1  Loading preprocessed NPZ
Features shape      : (23672, 140)
Participants        : 30
Sessions            : 837
Positive (0)        : 13,486 (57.0%)
Negative (1)        : 10,186 (43.0%)

STEP 2  Building splits
Subject-dependent (session-safe):
  train=15,283, val=3,669, test=4,720
Subject-independent (participant-safe):
  train=14,822, val=3,735, test=5,115
  Train participants=19, Test participants=6

STEP 3  Train and evaluate

----------------------------------------------------------------------
Subject-Dependent Session-Safe
----------------------------------------------------------------------
train class before: [8873 6410]
train class after : [8776 8776]
Tuning XGBoost with Optuna...


  0%|          | 0/25 [00:00<?, ?it/s]

best CV BA: 81.62%
Validation:
  BA   : 56.69%
  AUC  : 0.6067
Test:
  Acc  : 62.08%
  BA   : 61.70%
  AUC  : 0.6646
              precision    recall  f1-score   support

 Positive(0)       0.67      0.65      0.66      2655
 Negative(1)       0.56      0.59      0.58      2065

    accuracy                           0.62      4720
   macro avg       0.62      0.62      0.62      4720
weighted avg       0.62      0.62      0.62      4720


----------------------------------------------------------------------
Subject-Independent Participant-Safe
----------------------------------------------------------------------
train class before: [8454 6368]
train class after : [8294 8294]
Tuning XGBoost with Optuna...


  0%|          | 0/25 [00:00<?, ?it/s]

best CV BA: 82.09%
Validation:
  BA   : 62.39%
  AUC  : 0.6670
Test:
  Acc  : 60.74%
  BA   : 60.67%
  AUC  : 0.6563
              precision    recall  f1-score   support

 Positive(0)       0.68      0.61      0.64      2942
 Negative(1)       0.53      0.60      0.57      2173

    accuracy                           0.61      5115
   macro avg       0.60      0.61      0.60      5115
weighted avg       0.62      0.61      0.61      5115


STEP 4  Personalization simulation
subject  ba_before  ba_after best_adapter  n_cal_windows  n_eval_windows
  ID016   0.750167  0.741129     isotonic            304             551
  ID024   0.573230  0.663134     isotonic            302             555
  ID009   0.697984  0.660320     isotonic            311             540
  ID010   0.551380  0.533961     logistic            296             548
  ID028   0.498643  0.530025     isotonic            309             543
  ID018   0.472777  0.467381     isotonic            302             554

Mean BA 

KeyboardInterrupt: 

In [ ]:
"""
colab_train_neurosense.py
=========================
Paste this entire file into ONE Google Colab cell and run.

This script expects:
  /content/drive/MyDrive/neurosense_preprocessed/neurosense_features_256hz.npz

Important:
- For EEGNet training, the NPZ must include key: eeg_windows (N, 4, 1024).
- If your old NPZ has only handcrafted features, rerun local preprocessing:
    python preprocess_neurosense_local.py --dataset_root NeuroSenseEEGDataset/BIDS --out_dir neurosense_preprocessed

What it does:
1. Loads raw EEG windows + labels from NPZ.
2. Trains a compact EEGNet-style binary valence model.
3. Evaluates subject-dependent and subject-independent splits.
4. Applies Euclidean Alignment for shift reduction.
5. Simulates few-shot personalization using a tiny logistic adapter.
6. Adds real-time decision smoothing with confidence gating.
7. Saves deployable model artifacts.
"""

# =============================================================================
# INSTALL
# =============================================================================
import subprocess
import sys

BASE_PKGS = [
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "joblib",
]

for pkg in BASE_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=True)

try:
    import torch  # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "--quiet"], check=True)


# =============================================================================
# IMPORTS
# =============================================================================
import json
import copy
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F_torch
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_score,
)
from sklearn.model_selection import GroupShuffleSplit

from google.colab import drive
drive.mount("/content/drive")


# =============================================================================
# CONFIG
# =============================================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NPZ_PATH = Path("/content/drive/MyDrive/neurosense_preprocessed/neurosense_features_256hz.npz")
SAVE_DIR = Path("/content/drive/MyDrive/neurosense_trained_eegnet")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CAL_SESSIONS_PER_USER = 10
BATCH_SIZE = 256
EPOCHS = 30
PATIENCE = 6
LR = 1e-3
WEIGHT_DECAY = 1e-4

SMOOTH_WIN = 5
CONF_POS_THR = 0.65
CONF_NEG_THR = 0.35


# =============================================================================
# UTILITIES
# =============================================================================
def safe_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def class_weights_pos(y_bin: np.ndarray) -> float:
    n_pos = int((y_bin == 1).sum())
    n_neg = int((y_bin == 0).sum())
    if n_pos == 0:
        return 1.0
    return max(1.0, n_neg / max(1, n_pos))


def matrix_sqrt_spd(m: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    m = 0.5 * (m + m.T)
    vals, vecs = np.linalg.eigh(m)
    vals = np.clip(vals, eps, None)
    return (vecs * np.sqrt(vals)) @ vecs.T


def matrix_inv_sqrt_spd(m: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    m = 0.5 * (m + m.T)
    vals, vecs = np.linalg.eigh(m)
    vals = np.clip(vals, eps, None)
    return (vecs * (1.0 / np.sqrt(vals))) @ vecs.T


def window_cov(win: np.ndarray) -> np.ndarray:
    # win: (C, T)
    x = win - win.mean(axis=1, keepdims=True)
    cov = (x @ x.T) / max(1, x.shape[1] - 1)
    tr = float(np.trace(cov))
    if tr > 1e-12:
        cov = cov / tr
    return cov


def compute_reference_cov(wins: np.ndarray, max_samples: int = 12000) -> np.ndarray:
    n = len(wins)
    if n == 0:
        raise ValueError("No windows provided for reference covariance")

    if n > max_samples:
        idx = np.random.choice(n, size=max_samples, replace=False)
        sample = wins[idx]
    else:
        sample = wins

    cov_acc = np.zeros((wins.shape[1], wins.shape[1]), dtype=np.float64)
    for w in sample:
        cov_acc += window_cov(w.astype(np.float64))
    cov_ref = cov_acc / len(sample)
    cov_ref = 0.5 * (cov_ref + cov_ref.T)
    return cov_ref.astype(np.float64)


def euclidean_align_windows(wins: np.ndarray, cov_ref: np.ndarray) -> np.ndarray:
    # X' = sqrtm(cov_ref) @ invsqrtm(cov_win) @ X
    sqrt_ref = matrix_sqrt_spd(cov_ref)
    out = np.empty_like(wins, dtype=np.float32)
    for i in range(len(wins)):
        w = wins[i].astype(np.float64)
        cw = window_cov(w)
        inv_sqrt_cw = matrix_inv_sqrt_spd(cw)
        a = sqrt_ref @ inv_sqrt_cw
        out[i] = (a @ w).astype(np.float32)
    return out


def smooth_confident_states(probs_neg: np.ndarray, sessions: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Returns states {-1 uncertain, 0 positive, 1 negative} and mask decided
    states = np.full(len(probs_neg), -1, dtype=np.int8)

    for sess in np.unique(sessions):
        idx = np.where(sessions == sess)[0]
        p = probs_neg[idx]

        # Raw state with confidence gating
        raw = np.full(len(p), -1, dtype=np.int8)
        raw[p >= CONF_POS_THR] = 1
        raw[p <= CONF_NEG_THR] = 0

        # Majority smoothing on recent windows
        sm = np.full(len(p), -1, dtype=np.int8)
        for i in range(len(p)):
            lo = max(0, i - SMOOTH_WIN + 1)
            seg = raw[lo:i + 1]
            dec = seg[seg >= 0]
            if len(dec) == 0:
                continue
            c0 = int((dec == 0).sum())
            c1 = int((dec == 1).sum())
            if c0 >= 3:
                sm[i] = 0
            elif c1 >= 3:
                sm[i] = 1
        states[idx] = sm

    decided = states >= 0
    return states, decided


# =============================================================================
# DATASET / MODEL
# =============================================================================
class EEGWindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # X expected shape: (N, 4, 1024)
        self.X = torch.from_numpy(X[:, None, :, :].astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class CompactEEGNet(nn.Module):
    def __init__(self, n_ch: int = 4, n_samp: int = 1024, dropout: float = 0.4):
        super().__init__()

        # Temporal filtering
        self.conv_time = nn.Conv2d(1, 16, kernel_size=(1, 64), padding=(0, 32), bias=False)
        self.bn_time = nn.BatchNorm2d(16)

        # Depthwise spatial filter across channels
        self.conv_spatial = nn.Conv2d(16, 32, kernel_size=(n_ch, 1), groups=16, bias=False)
        self.bn_spatial = nn.BatchNorm2d(32)

        # Separable temporal block
        self.conv_sep_dw = nn.Conv2d(32, 32, kernel_size=(1, 16), padding=(0, 8), groups=32, bias=False)
        self.conv_sep_pw = nn.Conv2d(32, 32, kernel_size=(1, 1), bias=False)
        self.bn_sep = nn.BatchNorm2d(32)

        self.pool1 = nn.AvgPool2d(kernel_size=(1, 4))
        self.pool2 = nn.AvgPool2d(kernel_size=(1, 8))
        self.drop = nn.Dropout(dropout)

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_ch, n_samp)
            emb = self._forward_features(dummy)
            emb_dim = emb.shape[1]

        self.emb_dim = emb_dim
        self.head = nn.Linear(emb_dim, 1)

    def _forward_features(self, x):
        x = self.conv_time(x)
        x = self.bn_time(x)
        x = F_torch.elu(x)

        x = self.conv_spatial(x)
        x = self.bn_spatial(x)
        x = F_torch.elu(x)
        x = self.pool1(x)
        x = self.drop(x)

        x = self.conv_sep_dw(x)
        x = self.conv_sep_pw(x)
        x = self.bn_sep(x)
        x = F_torch.elu(x)
        x = self.pool2(x)
        x = self.drop(x)

        x = torch.flatten(x, start_dim=1)
        return x

    def forward(self, x):
        emb = self._forward_features(x)
        logits = self.head(emb).squeeze(1)
        return logits, emb


# =============================================================================
# TRAIN / EVAL HELPERS
# =============================================================================
def run_inference(model: nn.Module, X: np.ndarray, batch_size: int = 512):
    model.eval()
    ds = EEGWindowDataset(X, np.zeros(len(X), dtype=np.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    probs = []
    emb_all = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(DEVICE)
            logits, emb = model(xb)
            p = torch.sigmoid(logits).cpu().numpy()
            probs.append(p)
            emb_all.append(emb.cpu().numpy())

    return np.concatenate(probs), np.concatenate(emb_all)


def train_one_split(name: str, X: np.ndarray, y: np.ndarray, subjects: np.ndarray, sessions: np.ndarray,
                    idx_tr: np.ndarray, idx_va: np.ndarray, idx_te: np.ndarray):
    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    X_tr, y_tr = X[idx_tr], y[idx_tr]
    X_va, y_va = X[idx_va], y[idx_va]
    X_te, y_te = X[idx_te], y[idx_te]

    # Euclidean Alignment: reference from train only
    cov_ref = compute_reference_cov(X_tr)
    X_tr_al = euclidean_align_windows(X_tr, cov_ref)
    X_va_al = euclidean_align_windows(X_va, cov_ref)
    X_te_al = euclidean_align_windows(X_te, cov_ref)

    ds_tr = EEGWindowDataset(X_tr_al, y_tr)
    ds_va = EEGWindowDataset(X_va_al, y_va)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False)

    model = CompactEEGNet(n_ch=X.shape[1], n_samp=X.shape[2]).to(DEVICE)
    pos_w = class_weights_pos(y_tr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_state = None
    best_val_ba = -1.0
    bad_epochs = 0

    print(f"train class counts: {np.bincount(y_tr)}")
    print(f"val class counts  : {np.bincount(y_va)}")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0

        for xb, yb in dl_tr:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += float(loss.item()) * len(xb)

        # Validation BA for early stopping
        model.eval()
        va_prob = []
        va_true = []
        with torch.no_grad():
            for xb, yb in dl_va:
                xb = xb.to(DEVICE)
                logits, _ = model(xb)
                p = torch.sigmoid(logits).cpu().numpy()
                va_prob.append(p)
                va_true.append(yb.numpy())

        va_prob = np.concatenate(va_prob)
        va_true = np.concatenate(va_true).astype(np.int8)
        va_pred = (va_prob >= 0.5).astype(np.int8)
        va_ba = balanced_accuracy_score(va_true, va_pred)

        print(f"epoch {epoch:02d} | train_loss={loss_sum/len(ds_tr):.4f} | val_BA={va_ba*100:.2f}%")

        if va_ba > best_val_ba:
            best_val_ba = va_ba
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    if best_state is None:
        raise RuntimeError("No best model state captured during training")

    model.load_state_dict(best_state)

    # Final metrics
    va_prob, _ = run_inference(model, X_va_al)
    te_prob, te_emb = run_inference(model, X_te_al)

    va_pred = (va_prob >= 0.5).astype(np.int8)
    te_pred = (te_prob >= 0.5).astype(np.int8)

    va_ba = balanced_accuracy_score(y_va, va_pred)
    va_auc = safe_auc(y_va, va_prob)

    te_acc = float((te_pred == y_te).mean())
    te_ba = balanced_accuracy_score(y_te, te_pred)
    te_auc = safe_auc(y_te, te_prob)

    print("Validation:")
    print(f"  BA   : {va_ba*100:.2f}%")
    print(f"  AUC  : {va_auc:.4f}")
    print("Test:")
    print(f"  Acc  : {te_acc*100:.2f}%")
    print(f"  BA   : {te_ba*100:.2f}%")
    print(f"  AUC  : {te_auc:.4f}")
    print(classification_report(y_te, te_pred, target_names=["Positive(0)", "Negative(1)"]))

    cm = confusion_matrix(y_te, te_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["Pos", "Neg"], yticklabels=["Pos", "Neg"],
                cbar=False, ax=ax)
    ax.set_title(f"{name} - Test Confusion")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    fig.savefig(SAVE_DIR / f"cm_{name.replace(' ', '_').lower()}.png", dpi=150)
    plt.close(fig)

    return {
        "name": name,
        "model": model,
        "cov_ref": cov_ref,
        "val_ba": float(va_ba),
        "val_auc": float(va_auc),
        "test_acc": float(te_acc),
        "test_ba": float(te_ba),
        "test_auc": float(te_auc),
        "idx_test": idx_te,
        "y_test": y_te,
        "p_test": te_prob,
        "emb_test": te_emb,
        "sessions_test": sessions[idx_te],
        "subjects_test": subjects[idx_te],
    }


def personalization_with_tiny_head(res_ind: dict):
    y_te = res_ind["y_test"]
    p_te = res_ind["p_test"]
    emb_te = res_ind["emb_test"]
    sess_te = res_ind["sessions_test"]
    sub_te = res_ind["subjects_test"]

    rows = []
    for sub in np.unique(sub_te):
        m_sub = sub_te == sub
        if m_sub.sum() < 100:
            continue

        sess = sorted(np.unique(sess_te[m_sub]))
        if len(sess) <= CAL_SESSIONS_PER_USER:
            continue

        cal_sess = set(sess[:CAL_SESSIONS_PER_USER])
        eval_sess = set(sess[CAL_SESSIONS_PER_USER:])

        m_cal = m_sub & np.isin(sess_te, list(cal_sess))
        m_eval = m_sub & np.isin(sess_te, list(eval_sess))
        if m_cal.sum() < 30 or m_eval.sum() < 30:
            continue

        y_cal = y_te[m_cal]
        y_eval = y_te[m_eval]
        p_eval = p_te[m_eval]
        e_cal = emb_te[m_cal]
        e_eval = emb_te[m_eval]

        if len(np.unique(y_cal)) < 2 or len(np.unique(y_eval)) < 2:
            continue

        ba_before = balanced_accuracy_score(y_eval, (p_eval >= 0.5).astype(np.int8))

        # Tiny personalized head on embeddings (few-shot)
        tiny = LogisticRegression(
            max_iter=1000,
            C=0.5,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )
        tiny.fit(e_cal, y_cal)
        p_after = tiny.predict_proba(e_eval)[:, 1]
        ba_after = balanced_accuracy_score(y_eval, (p_after >= 0.5).astype(np.int8))

        rows.append({
            "subject": sub,
            "ba_before": float(ba_before),
            "ba_after": float(ba_after),
            "n_cal_windows": int(m_cal.sum()),
            "n_eval_windows": int(m_eval.sum()),
        })

    if not rows:
        print("Not enough eligible users for stable personalization simulation.")
        return None

    df = pd.DataFrame(rows).sort_values("ba_after", ascending=False)
    df.to_csv(SAVE_DIR / "personalization_tiny_head_per_user.csv", index=False)

    print(df.to_string(index=False))
    before_mean = df["ba_before"].mean() * 100.0
    after_mean = df["ba_after"].mean() * 100.0
    print(f"\nMean BA before: {before_mean:.2f}%")
    print(f"Mean BA after : {after_mean:.2f}%")
    print(f"Mean gain     : {after_mean - before_mean:+.2f}%")

    return {
        "table": df,
        "before_mean": float(before_mean),
        "after_mean": float(after_mean),
    }


def realtime_reliability(res_ind: dict):
    y_true = res_ind["y_test"]
    p_neg = res_ind["p_test"]
    sessions = res_ind["sessions_test"]

    states, decided = smooth_confident_states(p_neg, sessions)
    cov = float(decided.mean())

    if decided.sum() == 0:
        print("No decided windows under current confidence gates.")
        return {
            "coverage": 0.0,
            "precision": float("nan"),
            "ba": float("nan"),
        }

    y_dec = y_true[decided]
    s_dec = states[decided]

    ba = balanced_accuracy_score(y_dec, s_dec)
    prec = precision_score(y_dec, s_dec, average="macro", zero_division=0)

    print("Real-time reliability (smoothed, confidence-gated):")
    print(f"  Coverage   : {cov*100:.2f}% of windows")
    print(f"  Macro Prec : {prec*100:.2f}%")
    print(f"  BA         : {ba*100:.2f}%")

    return {
        "coverage": float(cov),
        "precision": float(prec),
        "ba": float(ba),
    }


# =============================================================================
# LOAD DATA
# =============================================================================
print("=" * 70)
print("STEP 1  Loading preprocessed NPZ")
print("=" * 70)

if not NPZ_PATH.exists():
    raise FileNotFoundError(f"NPZ not found: {NPZ_PATH}")

data = np.load(NPZ_PATH, allow_pickle=True)

if "eeg_windows" not in data:
    raise KeyError(
        "This NPZ does not contain 'eeg_windows'. Re-run local preprocessing with the latest preprocess_neurosense_local.py and upload the new NPZ."
    )

X = data["eeg_windows"].astype(np.float32)
if "labels" in data:
    y = data["labels"].astype(np.int8)
elif "y" in data:
    y = data["y"].astype(np.int8)
else:
    raise KeyError("Expected key 'labels' (or 'y') in NPZ")

if "subject_id" in data:
    subjects = data["subject_id"].astype(str)
elif "participant_id" in data:
    subjects = data["participant_id"].astype(str)
else:
    raise KeyError("Expected key 'subject_id' (or 'participant_id') in NPZ")

if "session_id" in data:
    sessions = data["session_id"].astype(str)
else:
    raise KeyError("Expected key 'session_id' in NPZ")

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

bc = np.bincount(y)
print(f"EEG windows shape   : {X.shape}")
print(f"Participants        : {len(np.unique(subjects))}")
print(f"Sessions            : {len(np.unique(sessions))}")
print(f"Positive (0)        : {bc[0]:,} ({bc[0]/len(y)*100:.1f}%)")
print(f"Negative (1)        : {bc[1]:,} ({bc[1]/len(y)*100:.1f}%)")
print(f"Device              : {DEVICE}")


# =============================================================================
# SPLITS
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2  Building splits")
print("=" * 70)

# Subject-dependent but session-safe
gss_session = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_dep, idx_test_dep = next(gss_session.split(X, y, groups=sessions))

gss_session_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_dep_rel, idx_val_dep_rel = next(
    gss_session_val.split(X[idx_train_dep], y[idx_train_dep], groups=sessions[idx_train_dep])
)
idx_train_dep_final = idx_train_dep[idx_train_dep_rel]
idx_val_dep_final = idx_train_dep[idx_val_dep_rel]

# Subject-independent
gss_sub = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_ind, idx_test_ind = next(gss_sub.split(X, y, groups=subjects))

gss_sub_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_train_ind_rel, idx_val_ind_rel = next(
    gss_sub_val.split(X[idx_train_ind], y[idx_train_ind], groups=subjects[idx_train_ind])
)
idx_train_ind_final = idx_train_ind[idx_train_ind_rel]
idx_val_ind_final = idx_train_ind[idx_val_ind_rel]

print("Subject-dependent (session-safe):")
print(f"  train={len(idx_train_dep_final):,}, val={len(idx_val_dep_final):,}, test={len(idx_test_dep):,}")
print("Subject-independent (participant-safe):")
print(f"  train={len(idx_train_ind_final):,}, val={len(idx_val_ind_final):,}, test={len(idx_test_ind):,}")
print(
    f"  Train participants={len(np.unique(subjects[idx_train_ind_final]))}, "
    f"Test participants={len(np.unique(subjects[idx_test_ind]))}"
)


# =============================================================================
# TRAIN / TEST
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3  Train and evaluate (Compact EEGNet)")
print("=" * 70)

res_dep = train_one_split(
    "Subject-Dependent Session-Safe",
    X,
    y,
    subjects,
    sessions,
    idx_train_dep_final,
    idx_val_dep_final,
    idx_test_dep,
)

res_ind = train_one_split(
    "Subject-Independent Participant-Safe",
    X,
    y,
    subjects,
    sessions,
    idx_train_ind_final,
    idx_val_ind_final,
    idx_test_ind,
)


# =============================================================================
# PERSONALIZATION
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4  Personalization simulation (few-shot tiny head)")
print("=" * 70)

perso = personalization_with_tiny_head(res_ind)


# =============================================================================
# REAL-TIME RELIABILITY
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5  Real-time reliability check")
print("=" * 70)

rt = realtime_reliability(res_ind)


# =============================================================================
# SAVE FINAL MODEL
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6  Save final model")
print("=" * 70)

# Retrain final global on all windows with participant-safe model settings
cov_ref_all = compute_reference_cov(X)
X_all_al = euclidean_align_windows(X, cov_ref_all)

ds_all = EEGWindowDataset(X_all_al, y)
dl_all = DataLoader(ds_all, batch_size=BATCH_SIZE, shuffle=True)

model_final = CompactEEGNet(n_ch=X.shape[1], n_samp=X.shape[2]).to(DEVICE)
pos_w_all = class_weights_pos(y)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w_all, device=DEVICE))
optimizer = torch.optim.AdamW(model_final.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

for epoch in range(1, 11):
    model_final.train()
    loss_sum = 0.0
    for xb, yb in dl_all:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model_final(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        loss_sum += float(loss.item()) * len(xb)
    print(f"final-train epoch {epoch:02d} | loss={loss_sum/len(ds_all):.4f}")

model_path = SAVE_DIR / "neurosense_eegnet_global.pt"
meta_path = SAVE_DIR / "neurosense_eegnet_metadata.json"
adapter_path = SAVE_DIR / "realtime_config.joblib"

state = {
    "model_state_dict": model_final.state_dict(),
    "channels_order": ["TP9", "AF7", "AF8", "TP10"],
    "input_shape": [int(X.shape[1]), int(X.shape[2])],
    "cov_ref": cov_ref_all.astype(np.float32),
    "seed": RANDOM_SEED,
}
torch.save(state, model_path)

meta = {
    "npz_path": str(NPZ_PATH),
    "n_windows": int(len(X)),
    "n_subjects": int(len(np.unique(subjects))),
    "n_sessions": int(len(np.unique(sessions))),
    "subject_dependent_test_ba": float(res_dep["test_ba"]),
    "subject_dependent_test_auc": float(res_dep["test_auc"]),
    "subject_independent_test_ba": float(res_ind["test_ba"]),
    "subject_independent_test_auc": float(res_ind["test_auc"]),
    "smoothing_window": SMOOTH_WIN,
    "confidence_positive_threshold": CONF_POS_THR,
    "confidence_negative_threshold": CONF_NEG_THR,
    "realtime_coverage": float(rt["coverage"]),
    "realtime_macro_precision": float(rt["precision"]),
    "realtime_ba": float(rt["ba"]),
}

if perso is not None:
    meta["personalization_before_mean_ba"] = float(perso["before_mean"] / 100.0)
    meta["personalization_after_mean_ba"] = float(perso["after_mean"] / 100.0)

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

joblib.dump(
    {
        "smooth_win": SMOOTH_WIN,
        "conf_pos_thr": CONF_POS_THR,
        "conf_neg_thr": CONF_NEG_THR,
        "label_map": {0: "positive", 1: "negative", -1: "uncertain"},
    },
    adapter_path,
)

print("Saved:")
for fp in [model_path, meta_path, adapter_path]:
    print(f"  {fp} ({fp.stat().st_size/1024:.1f} KB)")


# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print("Subject-dependent (session-safe):")
print(f"  Acc : {res_dep['test_acc']*100:.2f}%")
print(f"  BA  : {res_dep['test_ba']*100:.2f}%")
print(f"  AUC : {res_dep['test_auc']:.4f}")

print("\nSubject-independent (participant-safe, realistic):")
print(f"  Acc : {res_ind['test_acc']*100:.2f}%")
print(f"  BA  : {res_ind['test_ba']*100:.2f}%")
print(f"  AUC : {res_ind['test_auc']:.4f}")

if perso is not None:
    print("\nPersonalization (tiny head, 10 sessions):")
    print(f"  Before mean BA : {perso['before_mean']:.2f}%")
    print(f"  After  mean BA : {perso['after_mean']:.2f}%")
    print(f"  Mean gain      : {perso['after_mean'] - perso['before_mean']:+.2f}%")

print("\nReal-time reliability mode:")
print(f"  Coverage       : {rt['coverage']*100:.2f}%")
print(f"  Macro Precision: {rt['precision']*100:.2f}%")
print(f"  BA (decided)   : {rt['ba']*100:.2f}%")

print("\nDone.")


Mounted at /content/drive
STEP 1  Loading preprocessed NPZ
EEG windows shape   : (23672, 4, 1024)
Participants        : 30
Sessions            : 837
Positive (0)        : 13,486 (57.0%)
Negative (1)        : 10,186 (43.0%)
Device              : cpu

STEP 2  Building splits
Subject-dependent (session-safe):
  train=15,283, val=3,669, test=4,720
Subject-independent (participant-safe):
  train=14,822, val=3,735, test=5,115
  Train participants=19, Test participants=6

STEP 3  Train and evaluate (Compact EEGNet)

----------------------------------------------------------------------
Subject-Dependent Session-Safe
----------------------------------------------------------------------
train class counts: [8873 6410]
val class counts  : [1958 1711]
epoch 01 | train_loss=0.8113 | val_BA=51.74%
epoch 02 | train_loss=0.7926 | val_BA=52.26%
epoch 03 | train_loss=0.7828 | val_BA=55.62%
epoch 04 | train_loss=0.7753 | val_BA=55.60%
epoch 05 | train_loss=0.7729 | val_BA=57.26%
epoch 06 | train_loss=0

In [ ]:
"""
colab_train_neurosense.py
=========================
Paste this entire file into ONE Google Colab cell and run.

Expects:
  /content/drive/MyDrive/neurosense_preprocessed/neurosense_features_256hz.npz

What it does:
1.  Loads 128-feature windows + labels from NPZ.
2.  Builds subject-independent (participant-safe) split [primary] and
    subject-dependent (session-safe) split [secondary].
3.  Fits StandardScaler on train split only.
4.  SMOTE-Tomek oversampling on train split only.
5.  Optuna XGBoost hyperparameter search (50 trials).
6.  Stacked ensemble: XGBoost(tuned) + RF + LDA + SVC → Logistic meta-learner.
7.  Evaluates both splits with per-participant breakdown.
8.  Personalization simulation: per-user threshold search on first 5 calibration
    sessions, evaluated on remaining sessions.
9.  Saves model artifacts (ensemble + scaler + metadata).
"""

# =============================================================================
# INSTALL
# =============================================================================
import subprocess, sys

for pkg in [
    "numpy", "pandas", "scikit-learn", "matplotlib", "seaborn",
    "joblib", "xgboost", "optuna", "imbalanced-learn",
]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=True)


# =============================================================================
# IMPORTS
# =============================================================================
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold

from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from sklearn.base import clone as sk_clone

from google.colab import drive
drive.mount("/content/drive")


# =============================================================================
# CONFIG
# =============================================================================
RANDOM_SEED        = 42
NPZ_PATH           = Path("/content/drive/MyDrive/neurosense_preprocessed/neurosense_features_256hz.npz")
SAVE_DIR           = Path("/content/drive/MyDrive/neurosense_trained_ensemble")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_TRIALS      = 50
CAL_SESSIONS       = 5      # sessions used for per-user threshold calibration
MIN_CAL_WINDOWS    = 15     # minimum windows needed per class in calibration set

np.random.seed(RANDOM_SEED)


# =============================================================================
# HELPERS
# =============================================================================
def safe_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def print_section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


# =============================================================================
# LOAD DATA
# =============================================================================
print_section("STEP 1  Loading preprocessed NPZ")

if not NPZ_PATH.exists():
    raise FileNotFoundError(f"NPZ not found: {NPZ_PATH}")

data = np.load(NPZ_PATH, allow_pickle=True)

if "features" not in data:
    raise KeyError("NPZ missing 'features' key. Re-run preprocess_neurosense_local.py.")

X        = data["features"].astype(np.float32)
y        = data["labels"].astype(np.int8)
subjects = data["subject_id"].astype(str)
sessions = data["session_id"].astype(str)

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

bc = np.bincount(y)
print(f"Feature windows shape : {X.shape}")
print(f"Participants          : {len(np.unique(subjects))}")
print(f"Sessions              : {len(np.unique(sessions))}")
print(f"Positive (0)          : {bc[0]:,} ({bc[0]/len(y)*100:.1f}%)")
print(f"Negative (1)          : {bc[1]:,} ({bc[1]/len(y)*100:.1f}%)")


# =============================================================================
# SPLITS
# =============================================================================
print_section("STEP 2  Building splits")

# --- Subject-independent (participant-safe) — PRIMARY ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_tr_ind, idx_te_ind = next(gss.split(X, y, groups=subjects))

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_tr_rel, idx_va_rel = next(
    gss_val.split(X[idx_tr_ind], y[idx_tr_ind], groups=subjects[idx_tr_ind])
)
idx_tr_ind_f = idx_tr_ind[idx_tr_rel]
idx_va_ind_f = idx_tr_ind[idx_va_rel]

# --- Subject-dependent (session-safe) — SECONDARY ---
gss_dep = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_tr_dep, idx_te_dep = next(gss_dep.split(X, y, groups=sessions))

gss_dep_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
idx_tr_dep_rel, idx_va_dep_rel = next(
    gss_dep_val.split(X[idx_tr_dep], y[idx_tr_dep], groups=sessions[idx_tr_dep])
)
idx_tr_dep_f = idx_tr_dep[idx_tr_dep_rel]
idx_va_dep_f = idx_tr_dep[idx_va_dep_rel]

print("Subject-independent (participant-safe)  [PRIMARY]:")
print(f"  train={len(idx_tr_ind_f):,}  val={len(idx_va_ind_f):,}  test={len(idx_te_ind):,}")
print(f"  Train participants={len(np.unique(subjects[idx_tr_ind_f]))}  "
      f"Test participants={len(np.unique(subjects[idx_te_ind]))}")
print("\nSubject-dependent (session-safe)  [SECONDARY]:")
print(f"  train={len(idx_tr_dep_f):,}  val={len(idx_va_dep_f):,}  test={len(idx_te_dep):,}")


# =============================================================================
# OPTUNA XGBoost TUNING  (on subject-independent split)
# =============================================================================
print_section(f"STEP 3  Optuna XGBoost tuning  ({OPTUNA_TRIALS} trials)")

X_tr = X[idx_tr_ind_f]
y_tr = y[idx_tr_ind_f]
X_va = X[idx_va_ind_f]
y_va = y[idx_va_ind_f]
X_te = X[idx_te_ind]
y_te = y[idx_te_ind]

# Scale
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_va_sc = scaler.transform(X_va)
X_te_sc = scaler.transform(X_te)

# SMOTE-Tomek on training set only
print(f"Before SMOTE-Tomek : {np.bincount(y_tr)}")
smt = SMOTETomek(random_state=RANDOM_SEED)
X_tr_bal, y_tr_bal = smt.fit_resample(X_tr_sc, y_tr)
print(f"After  SMOTE-Tomek : {np.bincount(y_tr_bal)}")


def xgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1200),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "learning_rate":     trial.suggest_float("lr", 0.01, 0.2, log=True),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_lvl", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("lambda", 1e-4, 10.0, log=True),
        "gamma":             trial.suggest_float("gamma", 0.0, 1.0),
        "use_label_encoder": False,
        "eval_metric":       "logloss",
        "random_state":      RANDOM_SEED,
        "n_jobs":            -1,
    }
    model = XGBClassifier(**params)
    model.fit(X_tr_bal, y_tr_bal, eval_set=[(X_va_sc, y_va)], verbose=False)
    preds = model.predict(X_va_sc)
    return balanced_accuracy_score(y_va, preds)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best = study.best_params
print(f"\nBest trial BA : {study.best_value*100:.2f}%")
print(f"Best params   : {best}")

xgb_best = XGBClassifier(
    n_estimators      = best["n_estimators"],
    max_depth         = best["max_depth"],
    min_child_weight  = best["min_child_weight"],
    learning_rate     = best["lr"],
    subsample         = best["subsample"],
    colsample_bytree  = best["colsample"],
    colsample_bylevel = best["colsample_lvl"],
    reg_alpha         = best["alpha"],
    reg_lambda        = best["lambda"],
    gamma             = best["gamma"],
    use_label_encoder = False,
    eval_metric       = "logloss",
    random_state      = RANDOM_SEED,
    n_jobs            = -1,
)


# =============================================================================
# STACKED ENSEMBLE
# =============================================================================
print_section("STEP 4  Building stacked ensemble\n"
              "        Base : XGBoost(tuned) + RF + LDA + SVC\n"
              "        Meta : Logistic Regression (passthrough=True)")

base_estimators = [
    ("xgb", xgb_best),
    ("rf",  RandomForestClassifier(
                n_estimators=500, max_features="sqrt",
                class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)),
    ("lda", LinearDiscriminantAnalysis()),
    ("svc", CalibratedClassifierCV(
                LinearSVC(max_iter=2000, class_weight="balanced",
                          random_state=RANDOM_SEED), cv=3)),
]

stack = StackingClassifier(
    estimators       = base_estimators,
    final_estimator  = LogisticRegression(
                           max_iter=1000, class_weight="balanced",
                           random_state=RANDOM_SEED),
    passthrough      = True,
    cv               = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
    n_jobs           = -1,
)

print("  Fitting stacked ensemble on balanced train set ... ", end="", flush=True)
stack.fit(X_tr_bal, y_tr_bal)
print("done.")

va_pred = stack.predict(X_va_sc)
print(f"  Validation BA : {balanced_accuracy_score(y_va, va_pred)*100:.2f}%")


# =============================================================================
# EVALUATE — helper
# =============================================================================
def evaluate_split(name, X_te_s, y_te_s, subj_te):
    print(f"\n{'-'*70}")
    print(name)
    print(f"{'-'*70}")

    pred  = stack.predict(X_te_s)
    prob  = stack.predict_proba(X_te_s)[:, 1]

    acc  = float((pred == y_te_s).mean())
    ba   = balanced_accuracy_score(y_te_s, pred)
    auc  = safe_auc(y_te_s, prob)

    print(f"  Accuracy          : {acc*100:.2f}%")
    print(f"  Balanced Accuracy : {ba*100:.2f}%")
    print(f"  ROC-AUC           : {auc:.4f}")
    print(classification_report(y_te_s, pred,
                                target_names=["Positive(0)", "Negative(1)"]))

    # Per-participant breakdown
    rows = []
    for sub in sorted(np.unique(subj_te)):
        m   = subj_te == sub
        b   = balanced_accuracy_score(y_te_s[m], pred[m])
        tag = "✓" if b >= 0.75 else ("~" if b >= 0.65 else "✗")
        rows.append({"subject": sub, "windows": int(m.sum()), "ba": b, "tag": tag})
        print(f"    {tag}  {sub:>8}   windows={m.sum():>4}  BA={b*100:.1f}%")

    df_pp = pd.DataFrame(rows)
    mean_ba = df_pp["ba"].mean()
    print(f"\n  Mean per-participant BA : {mean_ba*100:.2f}%")
    print(f"  Std                     : {df_pp['ba'].std()*100:.2f}%")

    # Confusion matrix plot
    cm = confusion_matrix(y_te_s, pred, normalize="true")
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["Pos", "Neg"], yticklabels=["Pos", "Neg"],
                cbar=False, ax=ax)
    ax.set_title(f"{name}\nTest Confusion Matrix")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    fig.tight_layout()
    fig.savefig(SAVE_DIR / f"cm_{name.replace(' ', '_').lower()}.png", dpi=150)
    plt.close(fig)

    return {"acc": acc, "ba": ba, "auc": auc, "mean_pp_ba": float(mean_ba),
            "prob": prob, "pred": pred}


# =============================================================================
# STEP 5  Evaluate
# =============================================================================
print_section("STEP 5  Test-set evaluation")

# Subject-independent (primary)
res_ind = evaluate_split(
    "Subject-Independent (participant-safe)",
    X_te_sc, y_te,
    subjects[idx_te_ind],
)

# Subject-dependent (secondary) — re-use same scaler, no refitting
X_tr_dep_sc = scaler.transform(X[idx_tr_dep_f])
y_tr_dep     = y[idx_tr_dep_f]
X_te_dep_sc  = scaler.transform(X[idx_te_dep])
y_te_dep     = y[idx_te_dep]

smt2 = SMOTETomek(random_state=RANDOM_SEED)
X_tr_dep_bal, y_tr_dep_bal = smt2.fit_resample(X_tr_dep_sc, y_tr_dep)

# Refit stack on dep-train (shares same object — clone for independence)
stack_dep = sk_clone(stack)
stack_dep.fit(X_tr_dep_bal, y_tr_dep_bal)

res_dep = evaluate_split(
    "Subject-Dependent (session-safe)",
    X_te_dep_sc, y_te_dep,
    subjects[idx_te_dep],
)


# =============================================================================
# STEP 6  Personalization simulation — per-user threshold search
# =============================================================================
print_section("STEP 6  Personalization (per-user threshold on first 5 sessions)")

print(f"  Calibration sessions per user : {CAL_SESSIONS}")
print(f"  Min windows per class in cal  : {MIN_CAL_WINDOWS}")

prob_te  = res_ind["prob"]
subj_te  = subjects[idx_te_ind]
sess_te  = sessions[idx_te_ind]
y_te_all = y_te

rows = []
for sub in np.unique(subj_te):
    m_sub  = subj_te == sub
    sess_u = sorted(np.unique(sess_te[m_sub]))

    if len(sess_u) <= CAL_SESSIONS:
        continue

    cal_sess  = set(sess_u[:CAL_SESSIONS])
    eval_sess = set(sess_u[CAL_SESSIONS:])

    m_cal  = m_sub & np.isin(sess_te, list(cal_sess))
    m_eval = m_sub & np.isin(sess_te, list(eval_sess))

    if m_cal.sum() < MIN_CAL_WINDOWS * 2 or m_eval.sum() < MIN_CAL_WINDOWS:
        continue

    y_cal   = y_te_all[m_cal]
    p_cal   = prob_te[m_cal]
    y_eval  = y_te_all[m_eval]
    p_eval  = prob_te[m_eval]

    if len(np.unique(y_cal)) < 2 or len(np.unique(y_eval)) < 2:
        continue

    # BA before — default 0.5 threshold
    ba_before = balanced_accuracy_score(y_eval, (p_eval >= 0.5).astype(np.int8))

    # Find best threshold on calibration windows
    thresholds = np.linspace(0.2, 0.8, 61)
    best_thr   = 0.5
    best_cal_ba = -1.0
    for thr in thresholds:
        cal_ba = balanced_accuracy_score(y_cal, (p_cal >= thr).astype(np.int8))
        if cal_ba > best_cal_ba:
            best_cal_ba = cal_ba
            best_thr    = thr

    ba_after = balanced_accuracy_score(y_eval, (p_eval >= best_thr).astype(np.int8))

    rows.append({
        "subject":       sub,
        "ba_before":     float(ba_before),
        "ba_after":      float(ba_after),
        "best_threshold": float(best_thr),
        "n_cal":         int(m_cal.sum()),
        "n_eval":        int(m_eval.sum()),
    })

if rows:
    df_p = pd.DataFrame(rows).sort_values("ba_after", ascending=False)
    df_p.to_csv(SAVE_DIR / "personalization_threshold_per_user.csv", index=False)
    print(df_p.to_string(index=False))
    b_mean = df_p["ba_before"].mean() * 100
    a_mean = df_p["ba_after"].mean() * 100
    helped = int((df_p["ba_after"] > df_p["ba_before"]).sum())
    print(f"\n  Mean BA before  : {b_mean:.2f}%")
    print(f"  Mean BA after   : {a_mean:.2f}%")
    print(f"  Mean gain       : {a_mean - b_mean:+.2f}%")
    print(f"  Users helped    : {helped}/{len(df_p)}")
else:
    print("  Not enough eligible participants for personalization simulation.")
    df_p = None


# =============================================================================
# STEP 7  Save artifacts
# =============================================================================
print_section("STEP 7  Saving model artifacts")

# Retrain final model on ALL data (using subject-independent model & scaler)
X_all_sc  = scaler.transform(X)
smt_final = SMOTETomek(random_state=RANDOM_SEED)
X_all_bal, y_all_bal = smt_final.fit_resample(X_all_sc, y)

stack_final = sk_clone(stack)
print("  Fitting final model on all data ... ", end="", flush=True)
stack_final.fit(X_all_bal, y_all_bal)
print("done.")

model_path  = SAVE_DIR / "neurosense_ensemble_final.joblib"
scaler_path = SAVE_DIR / "neurosense_scaler.joblib"
meta_path   = SAVE_DIR / "neurosense_ensemble_metadata.json"

joblib.dump(stack_final, model_path)
joblib.dump(scaler,      scaler_path)

meta = {
    "n_features":                   int(X.shape[1]),
    "channels_order":               ["TP9", "AF7", "AF8", "TP10"],
    "normalization":                 "per_window_per_channel_zscore",
    "n_windows":                    int(len(X)),
    "n_subjects":                   int(len(np.unique(subjects))),
    "n_sessions":                   int(len(np.unique(sessions))),
    "subject_independent_test_ba":  float(res_ind["ba"]),
    "subject_independent_test_auc": float(res_ind["auc"]),
    "subject_dependent_test_ba":    float(res_dep["ba"]),
    "subject_dependent_test_auc":   float(res_dep["auc"]),
    "cal_sessions_for_threshold":   CAL_SESSIONS,
    "optuna_trials":                OPTUNA_TRIALS,
    "best_xgb_params":              best,
}
if df_p is not None:
    meta["personalization_before_mean_ba"] = float(b_mean / 100)
    meta["personalization_after_mean_ba"]  = float(a_mean / 100)
    meta["personalization_mean_gain_ba"]   = float((a_mean - b_mean) / 100)

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

for fp in [model_path, scaler_path, meta_path]:
    print(f"  Saved: {fp}  ({fp.stat().st_size/1024:.1f} KB)")


# =============================================================================
# FINAL SUMMARY
# =============================================================================
print_section("FINAL SUMMARY")
print(f"  Model              : XGBoost(Optuna) + RF + LDA + SVC → Logistic meta")
print(f"  Features           : {X.shape[1]} frequency-domain (per-window z-score)")
print(f"  SMOTE-Tomek        : yes (train split only)")
print(f"  Optuna trials      : {OPTUNA_TRIALS}")
print()
print(f"  Subject-independent (participant-safe, realistic):")
print(f"    Acc : {res_ind['acc']*100:.2f}%")
print(f"    BA  : {res_ind['ba']*100:.2f}%")
print(f"    AUC : {res_ind['auc']:.4f}")
print(f"    Mean per-participant BA : {res_ind['mean_pp_ba']*100:.2f}%")
print()
print(f"  Subject-dependent (session-safe):")
print(f"    Acc : {res_dep['acc']*100:.2f}%")
print(f"    BA  : {res_dep['ba']*100:.2f}%")
print(f"    AUC : {res_dep['auc']:.4f}")
print(f"    Mean per-participant BA : {res_dep['mean_pp_ba']*100:.2f}%")
if df_p is not None:
    print()
    print(f"  Personalization (threshold search, {CAL_SESSIONS} cal sessions):")
    print(f"    Before mean BA : {b_mean:.2f}%")
    print(f"    After  mean BA : {a_mean:.2f}%")
    print(f"    Mean gain      : {a_mean - b_mean:+.2f}%")
    print(f"    Users helped   : {helped}/{len(df_p)}")
print()
print("Done.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STEP 1  Loading preprocessed NPZ
Feature windows shape : (23672, 128)
Participants          : 30
Sessions              : 837
Positive (0)          : 13,486 (57.0%)
Negative (1)          : 10,186 (43.0%)

STEP 2  Building splits
Subject-independent (participant-safe)  [PRIMARY]:
  train=14,822  val=3,735  test=5,115
  Train participants=19  Test participants=6

Subject-dependent (session-safe)  [SECONDARY]:
  train=15,283  val=3,669  test=4,720

STEP 3  Optuna XGBoost tuning  (50 trials)
Before SMOTE-Tomek : [8454 6368]
After  SMOTE-Tomek : [8240 8240]


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:03:06] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:03:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:03:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:04:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


Best trial BA : 62.92%
Best params   : {'n_estimators': 486, 'max_depth': 8, 'min_child_weight': 5, 'lr': 0.011455356310090994, 'subsample': 0.7069028877185635, 'colsample': 0.5167929424672361, 'colsample_lvl': 0.8015112504670927, 'alpha': 9.551743658898735, 'lambda': 8.810389339671355, 'gamma': 0.6041297184693766}

STEP 4  Building stacked ensemble
        Base : XGBoost(tuned) + RF + LDA + SVC
        Meta : Logistic Regression (passthrough=True)
  Fitting stacked ensemble on balanced train set ... done.
  Validation BA : 59.85%

STEP 5  Test-set evaluation

----------------------------------------------------------------------
Subject-Independent (participant-safe)
----------------------------------------------------------------------
  Accuracy          : 61.37%
  Balanced Accuracy : 58.63%
  ROC-AUC           : 0.6382
              precision    recall  f1-score   support

 Positive(0)       0.64      0.77      0.70      2942
 Negative(1)       0.56      0.40      0.47      2173



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Not enough eligible participants for personalization simulation.

STEP 7  Saving model artifacts
  Fitting final model on all data ... done.
  Saved: /content/drive/MyDrive/neurosense_trained_ensemble/neurosense_ensemble_final.joblib  (256302.9 KB)
  Saved: /content/drive/MyDrive/neurosense_trained_ensemble/neurosense_scaler.joblib  (3.6 KB)
  Saved: /content/drive/MyDrive/neurosense_trained_ensemble/neurosense_ensemble_metadata.json  (0.8 KB)

FINAL SUMMARY
  Model              : XGBoost(Optuna) + RF + LDA + SVC → Logistic meta
  Features           : 128 frequency-domain (per-window z-score)
  SMOTE-Tomek        : yes (train split only)
  Optuna trials      : 50

  Subject-independent (participant-safe, realistic):
    Acc : 61.37%
    BA  : 58.63%
    AUC : 0.6382
    Mean per-participant BA : 58.64%

  Subject-dependent (session-safe):
    Acc : 84.89%
    BA  : 84.23%
    AUC : 0.9426
    Mean per-participant BA : 85.17%

Done.
